# Protein embeddings

This notebook collects protein sequence, protein structure, isoform, PPI, cross-species, and cytokine-style workflows under one entity tutorial. Use `BioEmbedder.embed(...)` for embedding generation, then use `embpy.tl` and `embpy.pl` for annotation-aware analysis.

Cytokine-specific examples can be dropped into the same pattern once the cytokine dataset is available.


In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd

from embpy import BioEmbedder, pl, tl

RUN_REAL_EMBEDDING = False
RANDOM_STATE = 7
rng = np.random.default_rng(RANDOM_STATE)

embedder = BioEmbedder(device="auto", organism="human")

proteins = ["P04637", "P00533", "P38398", "P01133", "P40763", "P05412", "P24385", "P05231"]
protein_symbols = ["TP53", "EGFR", "BRCA1", "EGF", "STAT3", "JUN", "CCND1", "IL6"]


## 1. Sequence and structure embeddings

ESM, ESM-C, ProtT5, and Boltz-2 are all requested through the same method. Structure models may require their dedicated pixi environment and structure/model-specific options.


In [ ]:
if RUN_REAL_EMBEDDING:
    esm_payload = embedder.embed(
        proteins,
        entity_type="protein",
        id_type="uniprot",
        model="esm2_650M",
        output="payload",
        key="X_protein_esm2_650M",
        pooling_strategy="mean",
    )

    prot_t5_payload = embedder.embed(
        proteins,
        entity_type="protein",
        id_type="uniprot",
        model="prot_t5_xl_half",
        output="payload",
        key="X_protein_prot_t5_xl_half",
    )

    # Structure embeddings live here too when Boltz-2 inputs are available.
    # boltz_payload = embedder.embed(
    #     proteins,
    #     entity_type="protein",
    #     id_type="uniprot",
    #     model="boltz2",
    #     output="payload",
    #     key="X_protein_boltz2",
    # )

    print(esm_payload["key"], esm_payload["n_entities"], esm_payload["n_dims"])
else:
    print("Set RUN_REAL_EMBEDDING=True to run protein sequence or structure models.")


## 2. Plot-ready protein AnnData

Rows are proteins. Annotation columns can come from UniProt/InterPro via `tl.annotate_proteins(...)`, cytokine metadata, or your own tables.


In [ ]:
def make_protein_demo(ids: list[str], symbols: list[str]) -> ad.AnnData:
    family = pd.Series(
        ["TF", "RTK", "DNA repair", "growth factor", "TF", "TF", "cell cycle", "cytokine"],
        index=ids,
        name="family",
    )
    location = pd.Series(
        ["nucleus", "membrane", "nucleus", "secreted", "nucleus", "nucleus", "nucleus", "secreted"],
        index=ids,
        name="location",
    )
    centers = {name: rng.normal(size=14) for name in family.unique()}
    X_esm = np.vstack([centers[family.loc[p]] + rng.normal(scale=0.2, size=14) for p in ids]).astype("float32")
    X_t5 = (X_esm @ rng.normal(size=(14, 10)) + rng.normal(scale=0.25, size=(len(ids), 10))).astype("float32")
    X_ppi = (X_esm @ rng.normal(size=(14, 8)) + rng.normal(scale=0.35, size=(len(ids), 8))).astype("float32")

    obs = pd.DataFrame({"uniprot": ids, "symbol": symbols, "family": family.values, "location": location.values}, index=symbols)
    out = ad.AnnData(X=np.zeros((len(ids), 1), dtype="float32"), obs=obs, var=pd.DataFrame(index=["placeholder"]))
    out.obsm["X_protein_esm2"] = X_esm
    out.obsm["X_protein_prott5"] = X_t5
    out.obsm["X_protein_ppi"] = X_ppi
    return out

protein_space = make_protein_demo(proteins, protein_symbols)
protein_space


## 3. Annotate proteins

Network access is optional. Keep it explicit so notebooks can render offline.


In [ ]:
RUN_REMOTE_ANNOTATION = False

if RUN_REMOTE_ANNOTATION:
    protein_space = tl.annotate_proteins(
        protein_space,
        column="uniprot",
        id_type="uniprot_id",
        sources=["function", "location", "domains", "go"],
        copy=True,
    )
else:
    print(protein_space.obs[["symbol", "family", "location"]])


## 4. Plot and compare protein embeddings


In [ ]:
pl.plot_embedding_space(
    protein_space,
    obsm_key="X_protein_esm2",
    method="pca",
    color="family",
    annotate=True,
    annotate_col="symbol",
    title="Protein sequence embedding by family",
)

pl.embedding_color_panel(
    protein_space,
    obsm_key="X_protein_esm2",
    method="pca",
    color_keys=["family", "location"],
    annotate=True,
    annotate_col="symbol",
)

pl.plot_similarity_heatmap(
    adata=protein_space,
    obsm_key="X_protein_esm2",
    label_col="symbol",
    title="ESM-2 protein similarity",
)


In [ ]:
_, mean_overlap = tl.compute_knn_overlap(protein_space, "X_protein_esm2", "X_protein_prott5", k=3)
print(f"Mean ESM/ProtT5 KNN overlap: {mean_overlap:.3f}")

pl.knn_overlap(protein_space, obsm_keys=["X_protein_esm2", "X_protein_prott5", "X_protein_ppi"], k=3)
pl.cross_embedding_correlation(protein_space, "X_protein_esm2", "X_protein_ppi")
pl.embedding_distributions(protein_space, obsm_keys=["X_protein_esm2", "X_protein_prott5"], n_dims=5)


## 5. Cross-species and cytokine extensions

Cross-species orthologs and cytokine panels are protein-level analyses. Generate embeddings as above, then add ortholog/cytokine metadata and reuse the same plotting functions.


In [ ]:
if RUN_REAL_EMBEDDING:
    # Example sketch. Replace with your ortholog or cytokine table.
    # cross = tl.build_cross_species_adata(...)
    # pl.plot_species_umap(cross, obsm_key="X_protein_esm2_650M")
    pass
